## <a name="feature_maps">2. 图像中的特征与特征图</a>

在计算机中，图片一般表示为3维的数组，即图片的长、宽以及单个像素点上的颜色值。颜色通常用红绿蓝3种三原色组合而成，因此，表示图片的三维数组的shape为(w, h, 3)。从3维空间看，长和宽代表尺寸，而3即代表深度，也就是说，图片可以看成由3个颜色通道组成，每个颜色通道的大小都和原图一致。在下面的代码中，我们随机从COCO数据集中读取一张图片，显示其数组的shape和不同颜色的通道。

首先，我们加载一张示例图片，显示其数组维度信息

In [ ]:
%matplotlib inline

from PIL import Image
import numpy as np
import json

src_img = Image.open('data/dog.jpg')
print(src_img)

从输出结果中可以看到，Image对象有size和mode两个属性，size为长x宽，mode为RGB，表示图片有RGB三个颜色通道。接下来，我们用Image对象构建一个numpy的数组，然后查看数组的shape，并显示颜色通道。

#### 显示RGB图片的颜色通道

In [ ]:
img_ary = np.array(src_img)
print(img_ary.shape)


import matplotlib.pyplot as plt

# 用matplot构建一个2x2的图框，显示RGB颜色通道和原始图形
for i in range(3):
    plt.subplot(1, 3, i+1)
    # 构建一个０填充的数组，然后只取原图的某一个通道
    channel = np.zeros(img_ary.shape, dtype='uint8')
    
    channel[:, :, i] = img_ary[:, :, i]
    plt.imshow(channel)
    plt.axis('off') #不显示坐标

rgb_channels_save_path = "02.rgb_channels.jpg"
plt.savefig(rgb_channels_save_path)
plt.show()

可以看到图片的3个颜色通道，由于是原图，每个颜色通道能看到的图像细节是一样的，并且都和原图相同。如果我们将图片进行卷积和池化的操作，降低图片的尺寸而增加其深度，这时图像的细节会变得模糊，但是我们仍然能分辨出图片大致的内容，也就是说，尽管损失了细节，但我们可以从更抽象的特征获知图片的内容。对图片进行卷积和池化操作后，我们可以得到更抽象的信息，如大概的形状，轮廓等等，图片深度的上的每一个数组，都代表了图片的某一个特征，因此输出的新的三维数组，我们称为图片的特征图。在下面的代码中，我们对图片进行处理，并显示其特征图。

#### 使用卷积层处理图片

In [ ]:
from keras.layers import Conv2D, MaxPool2D
from keras.models import Sequential

model = Sequential()
# 为model增加一个卷积层，输出数组的深度为12，filter的尺寸为2x2
model.add(Conv2D(12, (2, 2), input_shape=img_ary.shape))
# 在卷积层之后，增加一个池化层，filter的尺寸为3x3，进一步降低每一张特征的尺寸
model.add(MaxPool2D(3, 3))

# keras中，输入的数据要增加一个维度表示batch_size，即一个batch中处理多少张图片
# 这里我们使用numpy的expand_dims方法为原图增加一个维度，shape变为(1, w, h, 3)
input_image = np.expand_dims(img_ary, axis=0)
conv_img = model.predict(input_image)
print(conv_img.shape)

可以看到，经过卷积和池化之后，图片的尺寸变小，而深度增加到12，即有12个通道，每个通道上保存了更为抽象的特征信息，下面我们将12个通道显示出来。

#### 显示每层深度上的通道

In [ ]:
conv_img = np.squeeze(conv_img) # squeeze方法删除shape中大小为1的维度，图片的shape重新变为(w, h, 12)
print(conv_img.shape)
for i in range(conv_img.shape[2]):
    plt.subplot(3, 4, i+1)
    plt.title('channel{}'.format(i+1))
    plt.axis('off')
    plt.imshow(conv_img[:, :, i]) # 只取当前深度的数值，构成图片
channels_savepath = '02.channels.jpg'
plt.savefig(channels_savepath)
plt.show()

# 接下来，我们以完整的尺寸显示原图的一个特征和卷积后的一个特征，可以看到，图片变得模糊，信息变得抽象
plt.imshow(img_ary[:, :, 0])
plt.axis('off')
original_first_feature_savepath = '02.original_first_feature.jpg'
plt.savefig(original_first_feature_savepath)
plt.show()

plt.imshow(conv_img[:, :, 0])
plt.axis('off')
conv_first_feature_savepath = '02.conv_first_feature.jpg'
plt.savefig(conv_first_feature_savepath)
plt.show()

接下来，我们再对图片进行一次卷积操作，输出更抽象的信息

In [ ]:
import keras

model2 = keras.models.clone_model(model)
model2.add(Conv2D(12, (3, 3)))
model2.add(MaxPool2D((4, 4)))

conv_img = model2.predict(input_image)
conv_img = np.squeeze(conv_img)
print(conv_img.shape)


# 显示第二次卷积操作后的第一个通道
plt.imshow(conv_img[:, :, 0])
plt.axis('off')
feature_conv_twice_savepath = '02.feature_conv_twice.jpg'
plt.savefig(feature_conv_twice_savepath)
plt.show()

到这里，我们对图片的卷积处理以及处理后的特征图有了一个初步的认识。在Mask R-CNN框架中，backbone网络使用ResNet，ResNet是一个有5层卷积的网络结构，通过backbone，图片的特征图被提取出来，并使用COCO数据集进行训练，并用于下一阶段，RPN中RoI的相关处理。